# Adversarial examples: FGSM and PGD on ResNet-18

**CIS400 / CIS600 — Cybersecurity & AI.** Everything this notebook runs is written out in the cells below: the preprocessing, both attacks, and the driver. Nothing is cloned and nothing is `pip install`ed — Colab already ships PyTorch, torchvision, Pillow, and matplotlib. Read the cells top to bottom, then start changing them.

The only thing fetched from the network is the pretrained ResNet-18 checkpoint (about 45 MB) and one sample image.

**Threat model.** White-box and untargeted: we have the model's weights and its gradients, and we only try to make the top-1 label *change* — not to steer it to a chosen class. That is the weakest interesting attack goal, which is worth remembering when you read a paper reporting a success rate.

> Attacks here run against a local, public model on images you supply. Keep it that way: this is a teaching sandbox, not something to point at a service you do not own.

## 0 · Where the code will run

A GPU makes PGD's iterations quicker but nothing here needs one; on CPU the whole notebook still finishes in well under a minute. In Colab, *Runtime → Change runtime type → T4 GPU* if you want it.

In [ ]:
import torch, torchvision

print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("device     ", "cuda" if torch.cuda.is_available() else "cpu")

## 1 · The model, and where normalization happens

ResNet-18 expects each channel standardized by the ImageNet mean and standard deviation. Where you apply that matters for an attack. If normalization were part of preprocessing, the attack would be perturbing normalized values and $\varepsilon$ would mean something slightly different in each channel. So we keep the image in raw $[0,1]$ pixels and fold normalization into the forward pass:

$$f(x) = \mathrm{ResNet}\!\left(\frac{x - \mu}{\sigma}\right), \qquad x \in [0,1]^{3 \times 224 \times 224}$$

Now a budget of $\varepsilon = 0.02$ means what it looks like it means: no pixel channel moves by more than about $5/255$. Gradients still flow through the normalization to $x$, because it is just an affine map.

In [ ]:
from functools import lru_cache

import numpy as np
import torch
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS = ResNet18_Weights.DEFAULT
LABELS = WEIGHTS.meta["categories"]

# ImageNet channel statistics. Note where these are applied: the attacks work on
# raw [0, 1] pixels and normalization is folded into the forward pass below,
# not into preprocessing. That is deliberate — it keeps the epsilon budget
# denominated in units a student can see on screen (1/255 of a pixel channel)
# rather than in post-normalization units that differ per channel.
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


@lru_cache(maxsize=1)
def get_model() -> torch.nn.Module:
    """Download once, cache, and return an evaluation-mode ImageNet model."""
    return resnet18(weights=WEIGHTS).to(DEVICE).eval()


def image_to_tensor(image: Image.Image) -> torch.Tensor:
    """Apply the spatial part of ImageNet preprocessing, preserving pixel scale."""
    image = image.convert("RGB")
    image = TF.resize(image, 256, interpolation=InterpolationMode.BILINEAR)
    image = TF.center_crop(image, [224, 224])
    return TF.to_tensor(image).unsqueeze(0).to(DEVICE)


def logits_for(model: torch.nn.Module, pixels: torch.Tensor) -> torch.Tensor:
    """Normalize, then classify. Both attacks differentiate through this."""
    return model((pixels - MEAN) / STD)

## 2 · Reading a prediction

`prediction` turns logits into a top-1 class and a softmax confidence. Treat that confidence as a number the model reports, not as a probability the world owes you — a successful attack usually produces a *confidently* wrong answer, which is exactly why confidence is a poor detector of one.

In [ ]:
def prediction(logits: torch.Tensor) -> tuple[int, float]:
    """Return the top-1 class id and its softmax confidence."""
    probabilities = logits.softmax(dim=1)
    confidence, class_id = probabilities.max(dim=1)
    return class_id.item(), confidence.item()


@torch.no_grad()
def classify(model: torch.nn.Module, pixels: torch.Tensor) -> tuple[int, float]:
    """Top-1 (class id, confidence) for a batch of one image. Reporting only."""
    return prediction(logits_for(model, pixels))


def to_numpy_image(tensor: torch.Tensor) -> np.ndarray:
    """Convert a 1x3xHxW tensor in [0, 1] to an HxWx3 uint8 array."""
    array = tensor.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
    return np.uint8(np.clip(array * 255.0, 0, 255))

## 3 · FGSM — one step to the edge of the budget

Goodfellow et al. (2015). Take the loss $J(\theta, x, y)$ the network was trained to minimize, and move the *input* in whichever direction raises it:

$$x' = \mathrm{clip}_{[0,1]}\big(x + \varepsilon \cdot \mathrm{sign}(\nabla_x J(\theta, x, y))\big)$$

The `sign` is the whole trick. A gradient *step* would move furthest along the few pixels with the largest partial derivatives; taking only the sign spends the full $\varepsilon$ on **every** pixel at once. That is the right move when the constraint is $\lVert x' - x \rVert_\infty \le \varepsilon$, since under an $L_\infty$ budget the per-pixel spend is free — this is the corner of the $L_\infty$ ball that maximizes the first-order increase in loss.

One backward pass, and the linear approximation of $J$ it relies on is only good near $x$. That assumption is what PGD stops making.

In [ ]:
from collections.abc import Callable

import torch


def fgsm_attack(
    model: torch.nn.Module,
    pixels: torch.Tensor,
    label: int,
    epsilon: float,
    logits_for: Callable[[torch.nn.Module, torch.Tensor], torch.Tensor],
) -> tuple[torch.Tensor, torch.Tensor]:
    """Take one loss-maximizing step bounded by ``epsilon`` per pixel channel."""
    attacked = pixels.detach().clone().requires_grad_(True)
    target = torch.tensor([label], device=pixels.device)
    loss = torch.nn.functional.cross_entropy(logits_for(model, attacked), target)

    model.zero_grad(set_to_none=True)
    loss.backward()
    gradient = attacked.grad.detach()

    adversarial = attacked + epsilon * gradient.sign()
    adversarial = adversarial.clamp(0, 1).detach()
    return adversarial, gradient

## 4 · PGD — small steps, projected back each time

Madry et al. (2018). Iterate FGSM with a smaller step $\alpha$, and after every step project back into the $L_\infty$ ball so the total budget still holds:

$$x^{(t+1)} = \Pi_{B_\infty(x, \varepsilon)}\Big(x^{(t)} + \alpha \cdot \mathrm{sign}\big(\nabla_x J(\theta, x^{(t)}, y)\big)\Big)$$

The projection $\Pi$ is the two-sided clamp in the code below: clip to $[x - \varepsilon,\, x + \varepsilon]$, then to $[0,1]$ so the result is still an image. Because the gradient is recomputed at each $x^{(t)}$, PGD can follow curvature that FGSM's single linearization misses — with $\alpha < \varepsilon$ it searches inside the ball instead of jumping straight to a corner.

Cost is the honest trade: $T$ steps means $T$ forward and backward passes.

In [ ]:
from collections.abc import Callable

import torch


def pgd_attack(
    model: torch.nn.Module,
    pixels: torch.Tensor,
    label: int,
    epsilon: float,
    step_size: float,
    steps: int,
    logits_for: Callable[[torch.nn.Module, torch.Tensor], torch.Tensor],
) -> torch.Tensor:
    """Take repeated signed-gradient steps inside an L-infinity pixel budget."""
    original = pixels.detach()
    adversarial = original.clone()
    target = torch.tensor([label], device=pixels.device)

    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = torch.nn.functional.cross_entropy(
            logits_for(model, adversarial), target
        )
        gradient = torch.autograd.grad(loss, adversarial)[0]

        adversarial = adversarial.detach() + step_size * gradient.sign()
        lower = original - epsilon
        upper = original + epsilon
        adversarial = torch.maximum(torch.minimum(adversarial, upper), lower)
        adversarial = adversarial.clamp(0, 1)

    return adversarial.detach()

## 5 · Running both under one budget

Both attacks get the same $\varepsilon$, so the comparison is fair. Note the label being attacked is the model's **own clean prediction**, not ground truth: we are measuring whether the model can be moved off its answer, which is well-defined even for an image with no correct ImageNet class.

In [ ]:
from dataclasses import dataclass


@dataclass
class AttackResult:
    """One image attacked two ways, plus everything needed to report it."""

    epsilon: float
    pixels: torch.Tensor
    clean_id: int
    clean_confidence: float
    fgsm_pixels: torch.Tensor
    fgsm_id: int
    fgsm_confidence: float
    pgd_pixels: torch.Tensor
    pgd_id: int
    pgd_confidence: float

    @property
    def fgsm_changed(self) -> bool:
        """Did FGSM move the model off its original answer?"""
        return self.fgsm_id != self.clean_id

    @property
    def pgd_changed(self) -> bool:
        """Did PGD move the model off its original answer?"""
        return self.pgd_id != self.clean_id

    def amplified_delta(self) -> torch.Tensor:
        """PGD's perturbation rescaled around neutral gray so it is visible."""
        delta = self.pgd_pixels - self.pixels
        return (delta / (2 * max(self.epsilon, 1e-6)) + 0.5).clamp(0, 1)

    def linf(self, attacked: torch.Tensor) -> float:
        """Largest per-channel change. Must never exceed epsilon."""
        return (attacked - self.pixels).abs().max().item()


def run_attacks(
    pixels: torch.Tensor,
    epsilon: float = 0.02,
    pgd_steps: int = 10,
    pgd_step_size: float = 0.005,
) -> AttackResult:
    """Attack one preprocessed image with FGSM and PGD under the same budget.

    Both attacks are *untargeted* and take the model's own clean prediction as
    the label to move away from — not ground truth. So "success" here means
    "the top-1 label changed", which is the weakest useful notion of success.
    """
    model = get_model()
    clean_id, clean_confidence = classify(model, pixels)

    fgsm_pixels, _ = fgsm_attack(model, pixels, clean_id, epsilon, logits_for)
    fgsm_id, fgsm_confidence = classify(model, fgsm_pixels)

    pgd_pixels = pgd_attack(
        model, pixels, clean_id, epsilon, pgd_step_size, pgd_steps, logits_for
    )
    pgd_id, pgd_confidence = classify(model, pgd_pixels)

    return AttackResult(
        epsilon=float(epsilon),
        pixels=pixels,
        clean_id=clean_id,
        clean_confidence=clean_confidence,
        fgsm_pixels=fgsm_pixels,
        fgsm_id=fgsm_id,
        fgsm_confidence=fgsm_confidence,
        pgd_pixels=pgd_pixels,
        pgd_id=pgd_id,
        pgd_confidence=pgd_confidence,
    )

## 6 · Pick an image

Runs on a public sample by default so *Run all* works untouched. To attack your own image in Colab, set `USE_UPLOAD = True` and rerun this cell.

In [ ]:
import io
from urllib.request import urlopen

# A public sample so "Run all" works with no interaction. This is the only
# network fetch in the notebook besides the pretrained ResNet-18 weights.
SAMPLE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
USE_UPLOAD = False  # Set to True in Colab to attack your own image instead.


def load_image(use_upload: bool = USE_UPLOAD) -> Image.Image:
    """Return a PIL image: one you upload in Colab, or the public sample."""
    if use_upload:
        try:
            from google.colab import files  # Only present inside Colab.
        except ImportError:
            print("Not running in Colab — falling back to the sample image.")
        else:
            uploaded = files.upload()
            if uploaded:
                return Image.open(io.BytesIO(next(iter(uploaded.values()))))
            print("Nothing uploaded — falling back to the sample image.")
    return Image.open(io.BytesIO(urlopen(SAMPLE_URL).read()))


image = load_image()
pixels = image_to_tensor(image)
print("pixel tensor:", tuple(pixels.shape))
print(f"pixel range: [{pixels.min():.3f}, {pixels.max():.3f}]")

## 7 · Attack it

Four panels: the original, FGSM's result, PGD's result, and PGD's perturbation rescaled around neutral gray so you can see its structure. The printed $L_\infty$ is the largest change any channel actually took — check that it never exceeds $\varepsilon$. If it did, the attack would be cheating.

In [ ]:
import matplotlib.pyplot as plt


def show_result(result) -> None:
    """Four panels: the original, both attacks, and PGD's amplified noise."""
    panels = [
        (result.pixels, "Original",
         result.clean_id, result.clean_confidence),
        (result.fgsm_pixels, "After FGSM",
         result.fgsm_id, result.fgsm_confidence),
        (result.pgd_pixels, "After PGD",
         result.pgd_id, result.pgd_confidence),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.8))
    for axis, (tensor, stage, class_id, confidence) in zip(axes, panels):
        axis.imshow(to_numpy_image(tensor))
        axis.set_title(
            f"{stage}\n{LABELS[class_id]}\n{confidence:.1%}", fontsize=11
        )
        axis.axis("off")

    axes[3].imshow(to_numpy_image(result.amplified_delta()))
    axes[3].set_title(
        "PGD perturbation\n(amplified around gray)\n"
        f"true L-inf = {result.linf(result.pgd_pixels):.4f}",
        fontsize=11,
    )
    axes[3].axis("off")

    fig.suptitle(
        f"epsilon = {result.epsilon:.3f}  ({result.epsilon * 255:.1f}/255 per channel)",
        fontsize=13,
    )
    fig.tight_layout()
    plt.show()


result = run_attacks(pixels, epsilon=0.02, pgd_steps=10, pgd_step_size=0.005)
show_result(result)

## 8 · The number that actually matters

A single flipped image is a demo, not a result. The question to ask of any attack — in this notebook or in a paper — is **what fraction of which population of inputs does it break, at what budget?** The sweep below prints hits over a denominator at each $\varepsilon$, which with one image is an honest $n = 1$. Read those rows as anecdote, and note how small the budgets are: the interesting behavior on this image happens below $1/255$.

$\varepsilon = 0$ is the control: it must report $0$, because a zero-budget attack cannot change anything. If it ever doesn't, the harness is broken.

In [ ]:
def sweep(images, epsilons, pgd_steps: int = 10, pgd_step_size=None):
    """Attack every image at every budget, reporting hits over a denominator.

    ``pgd_step_size=None`` scales the step with the budget (eps/4), which keeps
    PGD searching *inside* the ball at every row. A fixed step size larger than
    epsilon would still be projected back — the budget always holds — but PGD
    would degenerate into bouncing between corners, which is FGSM with extra
    forward passes.
    """
    print(f"{'epsilon':>9} {'x/255':>7} {'FGSM':>11} {'PGD':>11}")
    print("-" * 42)
    rows = []
    for epsilon in epsilons:
        step = pgd_step_size if pgd_step_size else max(epsilon / 4, 1e-5)
        fgsm_hits = pgd_hits = 0
        for candidate in images:
            outcome = run_attacks(candidate, epsilon, pgd_steps, step)
            fgsm_hits += int(outcome.fgsm_changed)
            pgd_hits += int(outcome.pgd_changed)
        total = len(images)
        rows.append((epsilon, fgsm_hits, pgd_hits, total))
        print(
            f"{epsilon:9.4f} {epsilon * 255:7.2f} "
            f"{fgsm_hits:>6}/{total:<4} {pgd_hits:>6}/{total:<4}"
        )
    return rows


# One image is not an evaluation. See the note below the output.
rows = sweep([pixels], [0.0, 0.0002, 0.0005, 0.001, 0.002, 0.008, 0.02])

## 9 · Things worth trying

- **Find the floor.** Push $\varepsilon$ down until the label stops flipping. On the sample image it survives to roughly $0.001$ — about $0.3/255$, a change smaller than one step of an 8-bit pixel value, and far smaller than what re-saving the image as a JPEG would do to it. Sit with what that implies about how close the decision boundary runs to an ordinary photograph.
- **Watch the confidence, not just the label.** At the default settings PGD does not merely flip the Samoyed — it reports the wrong class at essentially $100\%$. Any defense that plans to screen out attacks by looking for low-confidence predictions has to explain this row.
- **Starve PGD.** Set `pgd_steps=1, pgd_step_size=epsilon` and you have reconstructed FGSM. Confirm the two produce the same image.
- **Get a real denominator.** Load 20–50 images, pass them all to `sweep`, and watch the FGSM and PGD curves separate. That separation is the actual claim the PGD paper makes.
- **Break the comparison on purpose.** Set `pgd_step_size` larger than `epsilon`. PGD still respects the budget — the projection sees to that — but it stops searching and just bounces between corners.
- **Then ask the defensive question.** Everything above assumed white-box gradient access. What would you have to take away from the attacker to stop it, and what would that cost the people using the model?

---

The polished Gradio version of this demo — sliders, upload panel, and an in-app code tutorial — lives beside this notebook in the course repository and runs locally with `python app.py`. This notebook and that app import the same attack code; the cells above were generated from it.